# LangChain을 이용한 답글 생성 자동화 


In [1]:
# Load environment variables in a file called .env
# API 키 설정 
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

# Check the key
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

API key found and looks good so far!
Google API Key exists and begins AI


In [2]:
#필수 라이브러리 설치 및 API 키 설정
import openai
import os
from langchain_openai import ChatOpenAI  # LangChain 라이브러리에서 ChatOpenAI 가져오기
# OpenAI API 호출 설정
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.1)
print("API 키 설정이 완료되었습니다.")


API 키 설정이 완료되었습니다.


# LangChain을 이용한 답글 생성 자동화

In [3]:
import pandas as pd
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI  # ✅ import 추가

reviews = pd.read_excel('reviews.xlsx')

response_prompt = """
당신은 이공간의 관리자입니다.
다음 리뷰에 대한 답변을 작성해 주세요.
리뷰에 대한 공감과 친절한 인사를 포함해 주세요.
마지막으로 재방문을 기원하는 메시지를 남겨 주세요.
리뷰: {review}
메시지:
"""

prompt_template = PromptTemplate(input_variables=["review"], template=response_prompt)

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.1)

def generate_responses(reviews, prompt_template):
    responses = []
    for review in reviews:
        prompt = prompt_template.format(review=review)
        response = llm.invoke(prompt)  # ✅ .invoke() 사용
        responses.append(response.content)  # ✅ .content로 텍스트 추출
    return responses

responses = generate_responses(reviews['Review'][:3].to_list(), prompt_template)
for response in responses:
    print("답글:", response)


답글: 안녕하세요, 소중한 리뷰 남겨주셔서 진심으로 감사드립니다! 제품과 배송 모두 만족하셨다니 저희도 매우 기쁩니다. 앞으로도 변함없이 좋은 품질과 서비스로 보답할 수 있도록 최선을 다하겠습니다. 다시 한 번 감사드리며, 언제든지 편하게 방문해 주세요. 다음에 또 뵙길 기대하겠습니다!
답글: 안녕하세요, 소중한 의견 남겨주셔서 감사합니다. 기대에 미치지 못해 실망을 드린 점 진심으로 사과드립니다. 사진과 실제 모습이 달라 불편을 드린 점 깊이 공감하며, 고객님의 말씀을 바탕으로 더욱 신경 써서 개선해 나가겠습니다. 혹시 환불 관련하여 도움이 필요하시면 언제든지 편하게 연락 주시길 바랍니다. 앞으로 더 나은 서비스로 찾아뵐 수 있도록 노력하겠으며, 다시 한 번 방문해 주시길 진심으로 기다리겠습니다. 감사합니다.
답글: 안녕하세요, 소중한 리뷰 남겨주셔서 진심으로 감사드립니다. 가격 대비 만족스러운 제품이라는 말씀에 저희도 큰 보람을 느낍니다. 앞으로도 좋은 품질과 서비스로 보답할 수 있도록 최선을 다하겠습니다. 다시 찾아주시는 날까지 항상 건강하시고 행복하시길 바랍니다. 재방문을 진심으로 기다리겠습니다!


## Few-Shot Prompting

이 코드에서는 Few-Shot Prompting 기법을 사용하여 예시를 제공하고, 유사한 질문에 대한 답변을 생성합니다.
 Few-Shot Prompt Template은 몇 가지 예시를 통해 모델이 새로운 질문에 대해 정확하게 응답할 수 있도록 도와줍니다.

In [10]:
import pandas as pd
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_openai import ChatOpenAI

# 데이터 불러오기
reviews = pd.read_excel('reviews.xlsx')

# 예제 데이터 설정 (상품 구매 리뷰 기반)
examples = [
    {
        "question": "리뷰: '상품이 사진과 똑같고 품질도 너무 좋아요! 빠른 배송에 포장도 꼼꼼했습니다.'",
        "answer": """
1. 상품 품질에 대한 의견은 긍정적인가요? 네.
2. 배송/포장에 대한 의견은 부정적인가요? 아니오.
3. 전반적인 의견은 긍정적인가요? 네.

따라서 최종 결론: 긍정적
""",
    },
    {
        "question": "리뷰: '배송이 너무 늦었고, 상품도 사진이랑 색상이 완전히 달랐어요. 환불 요청했습니다.'",
        "answer": """
1. 상품 품질에 대한 의견은 긍정적인가요? 아니오.
2. 배송/포장에 대한 의견은 부정적인가요? 네.
3. 전반적인 의견은 긍정적인가요? 아니오.

따라서 최종 결론: 부정적
""",
    },
    {
        "question": "리뷰: '가격 대비 품질이 훌륭해요. 재구매 의사 있습니다!'",
        "answer": """
1. 상품 품질에 대한 의견은 긍정적인가요? 네.
2. 배송/포장에 대한 의견은 부정적인가요? 언급 없음.
3. 전반적인 의견은 긍정적인가요? 네.

따라서 최종 결론: 긍정적
""",
    },
    {
        "question": "리뷰: '포장이 엉망이라 상품이 파손된 채로 왔어요. 고객센터도 연결이 안 됩니다.'",
        "answer": """
1. 상품 품질에 대한 의견은 긍정적인가요? 아니오.
2. 배송/포장에 대한 의견은 부정적인가요? 네.
3. 전반적인 의견은 긍정적인가요? 아니오.

따라서 최종 결론: 부정적
""",
    },
]

# 예제 프롬프트 설정
example_template = PromptTemplate(
    input_variables=["question", "answer"],
    template="질문: {question}\n{answer}"
)

# Few-Shot Prompt Template 설정
prompt_template = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_template,
    prefix="아래의 예시처럼 상품 구매 리뷰에서 긍정적 또는 부정적 요소를 판단하고, 최종 결론을 내리세요.\n판단 기준: 상품 품질, 배송/포장, 고객 서비스",
    suffix="리뷰: {input}",
    input_variables=["input"]
)

# LLM 초기화
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.1)

# reviews.xlsx의 리뷰 데이터를 순차적으로 분석
print("=" * 60)
for i, row in reviews.head(5).iterrows():
    review_text = row["Review"]
    actual_label = row["Label"]

    # 프롬프트 생성
    formatted_prompt = prompt_template.format(input=review_text)

    # LLM 호출
    response = llm.invoke(formatted_prompt)

    print(f"[{i+1}번 리뷰]")
    print(f"리뷰    : {review_text}")
    print(f"실제라벨: {'긍정적' if actual_label == 1 else '부정적'}")
    print(f"모델분석: {response.content.strip()}")
    print("=" * 60)

[1번 리뷰]
리뷰    : 이 제품 정말 최고예요! 배송도 빠르고 품질도 훌륭합니다.
실제라벨: 긍정적
모델분석: 1. 상품 품질에 대한 의견은 긍정적인가요? 네.  
2. 배송/포장에 대한 의견은 부정적인가요? 아니오.  
3. 전반적인 의견은 긍정적인가요? 네.  

따라서 최종 결론: 긍정적
[2번 리뷰]
리뷰    : 완전 실망이에요. 사진이랑 너무 달라요. 환불하고 싶어요.
실제라벨: 부정적
모델분석: 1. 상품 품질에 대한 의견은 긍정적인가요? 아니오. (사진과 다르다는 점에서 부정적)
2. 배송/포장에 대한 의견은 부정적인가요? 언급 없음.
3. 전반적인 의견은 긍정적인가요? 아니오. (완전 실망, 환불 의사 표현으로 부정적)

따라서 최종 결론: 부정적
[3번 리뷰]
리뷰    : 가격 대비 정말 좋은 제품입니다. 재구매 의사 있어요!
실제라벨: 긍정적
모델분석: 1. 상품 품질에 대한 의견은 긍정적인가요? 네.  
2. 배송/포장에 대한 의견은 부정적인가요? 언급 없음.  
3. 전반적인 의견은 긍정적인가요? 네.  

따라서 최종 결론: 긍정적
[4번 리뷰]
리뷰    : 배송이 너무 늦었고 포장도 엉망이었습니다.
실제라벨: 부정적
모델분석: 1. 상품 품질에 대한 의견은 긍정적인가요? 언급 없음.  
2. 배송/포장에 대한 의견은 부정적인가요? 네.  
3. 전반적인 의견은 긍정적인가요? 아니오.  

따라서 최종 결론: 부정적
[5번 리뷰]
리뷰    : 친절한 고객 서비스 덕분에 문제가 잘 해결됐습니다.
실제라벨: 긍정적
모델분석: 1. 상품 품질에 대한 의견은 긍정적인가요? 언급 없음.  
2. 배송/포장에 대한 의견은 부정적인가요? 언급 없음.  
3. 고객 서비스에 대한 의견은 긍정적인가요? 네.  
4. 전반적인 의견은 긍정적인가요? 네.  

따라서 최종 결론: 긍정적


# LangChain LCEL을 이용한 답글 생성 자동화

In [9]:
import pandas as pd
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 데이터 불러오기
reviews = pd.read_excel('reviews.xlsx')

# 예제 데이터 설정 (상품 구매 리뷰 기반)
examples = [
    {
        "question": "리뷰: '상품이 사진과 똑같고 품질도 너무 좋아요! 빠른 배송에 포장도 꼼꼼했습니다.'",
        "answer": """
1. 상품 품질에 대한 의견은 긍정적인가요? 네.
2. 배송/포장에 대한 의견은 부정적인가요? 아니오.
3. 전반적인 의견은 긍정적인가요? 네.

따라서 최종 결론: 긍정적
""",
    },
    {
        "question": "리뷰: '배송이 너무 늦었고, 상품도 사진이랑 색상이 완전히 달랐어요. 환불 요청했습니다.'",
        "answer": """
1. 상품 품질에 대한 의견은 긍정적인가요? 아니오.
2. 배송/포장에 대한 의견은 부정적인가요? 네.
3. 전반적인 의견은 긍정적인가요? 아니오.

따라서 최종 결론: 부정적
""",
    },
    {
        "question": "리뷰: '가격 대비 품질이 훌륭해요. 재구매 의사 있습니다!'",
        "answer": """
1. 상품 품질에 대한 의견은 긍정적인가요? 네.
2. 배송/포장에 대한 의견은 부정적인가요? 언급 없음.
3. 전반적인 의견은 긍정적인가요? 네.

따라서 최종 결론: 긍정적
""",
    },
    {
        "question": "리뷰: '포장이 엉망이라 상품이 파손된 채로 왔어요. 고객센터도 연결이 안 됩니다.'",
        "answer": """
1. 상품 품질에 대한 의견은 긍정적인가요? 아니오.
2. 배송/포장에 대한 의견은 부정적인가요? 네.
3. 전반적인 의견은 긍정적인가요? 아니오.

따라서 최종 결론: 부정적
""",
    },
]

# 예제 프롬프트 설정
example_template = PromptTemplate(
    input_variables=["question", "answer"],
    template="질문: {question}\n{answer}"
)

# Few-Shot Prompt Template 설정
prompt_template = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_template,
    prefix="아래의 예시처럼 상품 구매 리뷰에서 긍정적 또는 부정적 요소를 판단하고, 최종 결론을 내리세요.\n판단 기준: 상품 품질, 배송/포장, 고객 서비스",
    suffix="리뷰: {input}",
    input_variables=["input"]
)

# LLM 및 파서 초기화
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.1)
output_parser = StrOutputParser()

# LCEL | 연산자로 체인 구성
chain = prompt_template | llm | output_parser

# ✅ 상위 5개 리뷰만 분석
print("=" * 60)
for i, row in reviews.head(5).iterrows():
    review_text = row["Review"]
    actual_label = row["Label"]

    result = chain.invoke({"input": review_text})

    print(f"[{i+1}번 리뷰]")
    print(f"리뷰    : {review_text}")
    print(f"실제라벨: {'긍정적' if actual_label == 1 else '부정적'}")
    print(f"모델분석: {result.strip()}")
    print("=" * 60)

[1번 리뷰]
리뷰    : 이 제품 정말 최고예요! 배송도 빠르고 품질도 훌륭합니다.
실제라벨: 긍정적
모델분석: 1. 상품 품질에 대한 의견은 긍정적인가요? 네.  
2. 배송/포장에 대한 의견은 부정적인가요? 아니오.  
3. 전반적인 의견은 긍정적인가요? 네.  

따라서 최종 결론: 긍정적
[2번 리뷰]
리뷰    : 완전 실망이에요. 사진이랑 너무 달라요. 환불하고 싶어요.
실제라벨: 부정적
모델분석: 1. 상품 품질에 대한 의견은 긍정적인가요? 아니오. (사진과 다르다는 점에서 부정적)
2. 배송/포장에 대한 의견은 부정적인가요? 언급 없음.
3. 전반적인 의견은 긍정적인가요? 아니오. (완전 실망, 환불 의사 표현으로 부정적)

따라서 최종 결론: 부정적
[3번 리뷰]
리뷰    : 가격 대비 정말 좋은 제품입니다. 재구매 의사 있어요!
실제라벨: 긍정적
모델분석: 1. 상품 품질에 대한 의견은 긍정적인가요? 네.  
2. 배송/포장에 대한 의견은 부정적인가요? 언급 없음.  
3. 전반적인 의견은 긍정적인가요? 네.  

따라서 최종 결론: 긍정적
[4번 리뷰]
리뷰    : 배송이 너무 늦었고 포장도 엉망이었습니다.
실제라벨: 부정적
모델분석: 1. 상품 품질에 대한 의견은 긍정적인가요? 언급 없음.  
2. 배송/포장에 대한 의견은 부정적인가요? 네.  
3. 전반적인 의견은 긍정적인가요? 아니오.  

따라서 최종 결론: 부정적
[5번 리뷰]
리뷰    : 친절한 고객 서비스 덕분에 문제가 잘 해결됐습니다.
실제라벨: 긍정적
모델분석: 1. 상품 품질에 대한 의견은 긍정적인가요? 언급 없음.  
2. 배송/포장에 대한 의견은 부정적인가요? 언급 없음.  
3. 고객 서비스에 대한 의견은 긍정적인가요? 네.  
4. 전반적인 의견은 긍정적인가요? 네.  

따라서 최종 결론: 긍정적
